Firstly, let's import every package we will need in this analysis. It is required to install 
```Python
%pip install atlasopenmagic
```
in order to open ATLAS Open Data

In [88]:
import numpy as np
from numpy.lib.recfunctions import structured_to_unstructured   # to transform awk to linear numpy array
import pandas as pd                 
import uproot                       # to open .root files
import awkward as ak                # to read data with uproot
import matplotlib.pyplot as plt     # to plot
import vector                       # allows to manipulate Lorentz vectors
import os                           # to manage directories
import random                       # extract random numbers
import requests                     # for HTTP access
import aiohttp                      # HTTP client support
import atlasopenmagic as atom       # to access ATLAS Open Data directly

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

tf.keras.mixed_precision.set_global_policy('mixed_float16')

## Import Data
Now we can download the dataset, selecting the release of interest. It is possible to see every release using
```Python
atom.available_releases()
```
We will use the 2025 release of data taken at $\sqrt{s}=13$ TeV in p-p collisions


In [2]:
atom.set_release("2025e-13tev-beta")    # select release
skim = "GamGam"                         # select skim: events with 2 photons
random.seed(24)                         # set seed for random extractions

# get keys for every dataset: Run2 datas have key="data", MC simulations have key=numbers
all_keys = atom.available_datasets()

run_url_list = atom.get_urls("data", skim, protocol="https", cache=True)      # get list of urls for Run2 data

# get list of urls for MC simulations
mc_url_list = []
for key in all_keys:
    if key != "data":
        mc_url_list += atom.get_urls(key, skim, protocol="https", cache=True)


print(f"Number of MonteCarlo simulated datasets: {len(mc_url_list)}")
print(f"Number of Run2 datasets: {len(run_url_list)}")

Fetching metadata for release: 2025e-13tev-beta...
Fetching datasets: 100%|██████████| 374/374 [00:00<00:00, 949.29datasets/s]
✓ Successfully cached 374 datasets.
Active release: 2025e-13tev-beta. (Datasets path: REMOTE)


Number of MonteCarlo simulated datasets: 373
Number of Run2 datasets: 16


MC simulated datasets are produced considering only one process in each dataset. To separate signal datasets from background datasets we have to select urls that contain the Higgs boson decay channel of interest: into two photons.
Its notation in *gamgam* or *yy*.

In [3]:
# select signal datasets: may contain "gamgam" or "yy"
signal_url_list = [url for url in mc_url_list if ("gamgam" in url) or ("Hyy" in url) or (("_yy" in url) and ("yyy" not in url))]

# select background datasets: all dataset not taken in signal
bkg_url_list = [url for url in mc_url_list if url not in signal_url_list]

print("Number of Signal datasets: ", len(signal_url_list))
print("Number of Background datasets: ", len(bkg_url_list))

Number of Signal datasets:  20
Number of Background datasets:  353


Now we open data in TTrees. In order to have a way to train rapidly the model, it is possible to switch between a subset of data and the entire dataset with a boolean `bool useall`

In [4]:
useall = False      # True: use all dataset
                    # False: use a subset

Explore content of datasets: TTree and TBranches

In [5]:
# See names of trees and branches
print("Tree name: ", uproot.open(mc_url_list[0]).keys())
print("Branches: ", uproot.open(f"{mc_url_list[0]}:analysis").keys())

Tree name:  ['analysis;1']
Branches:  ['sig_ph', 'n_sig_ph', 'num_events', 'sum_of_weights', 'sum_of_weights_squared', 'xsec', 'kfac', 'filteff', 'TriggerMatch_DILEPTON', 'ScaleFactor_MLTRIGGER', 'ScaleFactor_PILEUP', 'ScaleFactor_FTAG', 'mcWeight', 'channelNumber', 'eventNumber', 'runNumber', 'trigML', 'trigP', 'trigDT', 'trigT', 'trigE', 'trigDM', 'trigDE', 'trigM', 'trigMET', 'ScaleFactor_BTAG', 'ScaleFactor_JVT', 'jet_n', 'jet_pt', 'jet_eta', 'jet_phi', 'jet_e', 'jet_btag_quantile', 'jet_jvt', 'largeRJet_n', 'largeRJet_pt', 'largeRJet_eta', 'largeRJet_phi', 'largeRJet_e', 'largeRJet_m', 'largeRJet_D2', 'jet_pt_jer1', 'jet_pt_jer2', 'ScaleFactor_ELE', 'ScaleFactor_MUON', 'ScaleFactor_LepTRIGGER', 'ScaleFactor_MuTRIGGER', 'ScaleFactor_ElTRIGGER', 'lep_n', 'lep_type', 'lep_pt', 'lep_eta', 'lep_phi', 'lep_e', 'lep_charge', 'lep_ptvarcone30', 'lep_topoetcone20', 'lep_z0', 'lep_d0', 'lep_d0sig', 'lep_isTightID', 'lep_isMediumID', 'lep_isLooseID', 'lep_isTightIso', 'lep_isLooseIso', 'lep_

Define TTree name and TBranches that we want to extract for this analysis

In [6]:
tree = "analysis"           # name of TTree in ATLAS OD

features = [branch for branch in (uproot.open(f"{mc_url_list[0]}:{tree}").keys()) 
            if ("photon_" in branch) and ("truth" not in branch)]

#if not useall:
#        mc_ind = set(random.sample(range(len(mc_url_list)), 16))
#        mc_url_list = [url for i, url in enumerate(mc_url_list) if i in mc_ind]
        


Read datasets into awkward arrays. This procedure avoids errors in calling CERN server and it is the faster solution.
Then, label=1 is assigned to signal events, label=0 is assigned to background events.

**This download may last up to 15-20 minutes!**
Then, it is possible to save locally the awkward arrays and import them rapidly, without download

In [ ]:
# read signal datasets into awkward array
signal_array = []
for url in signal_url_list:    
    with uproot.open(f"{url}:{tree}") as tree_opened:
        arr = tree_opened.arrays(filter_name=features, library="ak")
        signal_array.append(arr)
signal_awk = ak.concatenate(signal_array)

# assign label 1 to signal
signal_awk["label"] = 1


# read background datasets into awkward array
bkg_array = []
for url in bkg_url_list:
    with uproot.open(f"{url}:{tree}") as tree_opened:
        arr = tree_opened.arrays(filter_name=features, library="ak")
        bkg_array.append(arr)
bkg_awk = ak.concatenate(bkg_array)

# assign label 0 to background
bkg_awk["label"] = 0            


# merge signal and background arrays into one
mc_awk = ak.concatenate([signal_awk, bkg_awk])

# create directory if needed and save array locally
os.makedirs("data", exist_ok=True)
ak.to_parquet(mc_awk, "data/mc_awkward_complete.parquet") 
print("MC awkward array correctly saved!")

Awkward array correctly saved!


In [7]:
# recover mc and run2 awkward arrays from local memory
mc_awk = ak.from_parquet("data/mc_awkward_complete.parquet")

print("MC awkward array correctly recovered!")

MC awkward arrays correctly recovered!


# Preprocessing Dataset
Now that we have the complete dataset, we need to preprocess for the Neural Network input.

Firstly, for each event, we order photons for decreasing $p_T$ in order to select the **leading** (index 0) and **subleading** (index 1) photons. These are the photons of interest for this analysis.

To do so, we need to separate *jagged* variables, that are associated to each photon, from *scalar* variables, that refers to the entire event

In [ ]:
# define function that sorts photons by pT: we will also use it for run2 dataset
def sort_photons_pt(array):
    # separate jagged and scalar var
    scalar_var = ["photon_n", "label"]
    jagged_var = [var for var in features if var not in scalar_var]

    # find indces of ordered photon, from higher pT to lower pT
    sorted_index = ak.argsort(array.photon_pt, axis=-1, ascending=False)
    
    # sort mc array with dictionary unpacking
    return ak.Array({
        **{var: array[var][sorted_index] for var in jagged_var},
        **{var: array[var] for var in scalar_var},
    })

In [9]:
# sort MC array
mc_sort = sort_photons_pt(mc_awk)

# check that photons have been ordered
n_events_notord = len(mc_sort[mc_sort["photon_pt",:,0]<mc_sort["photon_pt",:,1]])
if n_events_notord == 0:
    print("Every event has been properly sorted by pT!")
else:
    print(f"Error: there are {n_events_notord} events that are not sorted")

Every event has been properly sorted by pT!


Last check: we want to select only photons that are well reconstructed and isolated. To do so, we impose that leading and subleading photons respect the 4 boolean conditions in the dataset.

In [10]:
# define a function that selects only well reconstructed and isolated photons
def select_reconst_and_isolated_phtons(array):
    # make mask for those conditions
    mask = (
        array.photon_isLooseID[:, 0] & array.photon_isLooseID[:, 1] &
        array.photon_isTightID[:, 0] & array.photon_isTightID[:, 1] &
        array.photon_isLooseIso[:, 0] & array.photon_isLooseIso[:, 1] &
        array.photon_isTightIso[:, 0] & array.photon_isTightIso[:, 1]
    )
    # apply mask to array
    return array[mask]

In [11]:
# filter MC dataset: select only events with well reconstructed and isolated photons
mc_filtered = select_reconst_and_isolated_phtons(mc_sort)

### Invariant Mass $m_{\gamma\gamma}$
Now it is useful to calculate the invariant mass of leading and subleading photons for each event. In natural units, the invariant mass is defined as
$$ m_{\gamma\gamma} = \sqrt{E^2_\text{tot}-(\vec{p}_\text{tot})^2} $$
We can calculate it using `.M` methon in `vector` library, using 4-momentum-like vectors defined as 
$$ p_4 = (E, p_T, \phi, \eta) $$
The result is given in GeV

In [12]:
# define a function to calculate invariant mass
def invariant_mass_calc(array):
    # build 4-momentum for each photon
    p4 = vector.zip({
        "pt": array.photon_pt,
        "eta": array.photon_eta,
        "phi": array.photon_phi,
        "e": array.photon_e
    })
    return (p4[:, 0] + p4[:, 1]).M

In [13]:
# add invariant mass column in MC awkward array 
mc_filtered["photon_invariant_mass"] = invariant_mass_calc(mc_filtered)

# remove events with invariant mass = 0, as later we have to divide for it
mc_final = mc_filtered[mc_filtered["photon_invariant_mass"]!=0]

### Features Extraction
Now that we have everything we need, let's proceed to extract features with some precautions.

- $p_T$: in order to avoid the NN to learn $m_{\gamma\gamma}$ dependence, we normalize $p_T$ for it
- $E_\gamma$: same procedure
- $\eta$: is already adimensional and $m_{\gamma\gamma}$-independent
- $\phi$: to avoid differences of $-\pi$ and $+\pi$, we transform it into two features: $\cos\phi$ and $\sin\phi$
- ptcone20: for same reasons as $p_T$, normalize wrt $p_T$ to get relative isolation
- topoetcone40: same as previous

In [89]:
# define a function that separates features for leading and subleading photon
def get_features(array):
    """
    Input: awkward array, that is the filtered dataset
    Output: one awkward array containing features for leading (l) and for subleading (s) photons, properly normalized
    """
    # get leading photon features
    lead_features = {
        "pt_l": array.photon_pt[:, 0] / array.photon_invariant_mass,
        "eta_l": array.photon_eta[:, 0],
        "cos_phi_l": np.cos(array.photon_phi[:, 0]),
        "sin_phi_l": np.sin(array.photon_phi[:, 0]),
        "e_l": array.photon_e[:, 0] / array.photon_invariant_mass,
        "ptcone20_l": array.photon_ptcone20[:, 0] / array.photon_pt[:, 0],
        "topoetcone40_l": array.photon_topoetcone40[:, 0] / array.photon_pt[:, 0]
    }

    # get subleading photon features
    sublead_features = {
            "pt_s": array.photon_pt[:, 1] / array.photon_invariant_mass,
            "eta_s": array.photon_eta[:, 1],
            "cos_phi_s": np.cos(array.photon_phi[:, 1]),
            "sin_phi_s": np.sin(array.photon_phi[:, 1]),
            "e_s": array.photon_e[:, 1] / array.photon_invariant_mass,
            "ptcone20_s": array.photon_ptcone20[:, 1] / array.photon_pt[:, 1],
            "topoetcone40_s": array.photon_topoetcone40[:, 1] / array.photon_pt[:, 1]
        }

    # merge dictionaries and transform into awkward arrays
    return ak.Array({**lead_features, **sublead_features})

In [26]:
# obtain features for MC dataset
mc_X_tot= get_features(mc_final)


Now that we have obtained all the features we need for leading and subleading photons, we want to split the dataset into *training dataset* and *test dataset*.

Then, some features have to be standardized before training, in order to keep their values centered in zero. Note that $\sin\phi$ and $\cos\phi$ doesn't need to be standardized. This procedure consists of the following transformation:
$$ \hat{x} = \frac{x-\mu}{\sigma} $$
where $\mu$ and $\sigma$ are the mean and the standard deviation of the **TRAIN** dataset of the feature $x$.

This is important: in order to train properly the NN, the transformation have to be the same during training and during test.


In [96]:
def get_train_test_datasets(array):
    """
    Gets the filtered dataset as awkward array, separates features for leading and subleading
    photon and normalizes them properly, using get_features function. Then, splits dataset into train and test
    and returns linearized (numpy) arrays of, respectively, features and labels for training and features and labels for test
    """

    # use previously defined function
    X_array = get_features(array)

    n_events = len(X_array)               # get number of events  
    indices = np.arange(n_events)         # get array of indices
    # split dataset, get arrays of indeces for train and test datasets
    train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

    # arrays for training
    X_train = X_array[train_idx]        # features
    y_train = ak.to_numpy(array["label", train_idx])      # labels, to_numpy per linearizzare

    # arrays for test
    X_test = X_array[test_idx]          # features
    y_test = ak.to_numpy(array["label", test_idx])          # labels

    # select features to standardize
    features_to_scale = [field for field in X_train.fields if "phi" not in field]
    features_not_to_scale = [field for field in X_train.fields if "phi" in field]

    # calcuate mean and std dev on train dataset, for features we want to standardize, and construct two dictionaries
    means = {feature: ak.mean(X_train[feature]) for feature in features_to_scale}
    std_devs = {feature: ak.std(X_train[feature]) for feature in features_to_scale}
    
    
    # scale features
    X_train_scaled_struct = ak.to_numpy(ak.Array({
        **{feature: (X_train[feature] - means[feature]) / std_devs[feature] for feature in features_to_scale},
        **{feature: X_train[feature] for feature in features_not_to_scale}
    }))
    X_test_scaled_struct = ak.to_numpy(ak.Array({
            **{feature: (X_test[feature] - means[feature]) / std_devs[feature] for feature in features_to_scale},
            **{feature: X_test[feature] for feature in features_not_to_scale}
    }))

    X_train_scaled = structured_to_unstructured(X_train_scaled_struct).astype(np.float32)
    X_test_scaled = structured_to_unstructured(X_test_scaled_struct).astype(np.float32)

    return X_train_scaled, y_train, X_test_scaled, y_test, means, std_devs

In [97]:
X_train, y_train, X_test , y_test, means, std_devs  = get_train_test_datasets(mc_final)


Now the dataset is ready!

We can now resume the preprocessing steps in a single funtion that includes all preprocessing functions

In [98]:
def build_dataset(array, means=None, std_devs=None):
    """
    Important! Explicit means and std_dev only if dataset is run data! They have to be those used during training


    Resume function to preprocess dataset:
    1) Sorts photons by pT
    2) Filters dataset from events whose photons are not well reconstructed or not well isolated
    3) Calculates invariant mass of leading and subleading photon, for each event
    4) Filters dataset from event whose invariant mass is zero
    5) Separates features for leading and subleading photon, then normalizes them properly
    6) Standardizes features that need it

    If the dataset is a MC simulation, returns 6 arrays and that are X_train, y_train, X_test, y_test, means, std_devs
    If the dataset is Run data, returns 1 array that is X (features of two photons, whitout training/test split)
    """

    # check if dataset is simulation or run data
    is_mc = "label" in array.fields

    # 1) Sort photons by pT
    array_sorted = sort_photons_pt(array)

    # 2) Filters dataset from events whose photons are not well reconstructed or not well isolated
    array_filtered = select_reconst_and_isolated_phtons(array_sorted)

    # 3) Calculates invariant mass of leading and subleading photon, for each event
    array_filtered["photon_invariant_mass"] = invariant_mass_calc(array_filtered)

    # 4) Filters dataset from event whose invariant mass is zero
    array_final = array_filtered[array_filtered["photon_invariant_mass"]!=0]

    # 5) Separates features for leading and subleading photon, then normalizes them properly
    # 6) Standardizes features that need it

    if is_mc:
        return get_train_test_datasets(array_final)
    else:               # if is Run data, we need mean and std_devs from the train dataset, to normalize
        if (means==None) and (std_devs==None):
            raise ValueError("Means and Standard Deviations are not specified for Run dataset! Specify them")
        else:
            X = get_features(array_final)

            # standardize features with training mean and std dev values
            X_scaled = ak.to_numpy(ak.Array({
                            **{feature: (X[feature] - means[feature]) / std_devs[feature] for feature in features_to_scale},
                            **{feature: X[feature] for feature in features_not_to_scale}
                    }))
            return X_scaled


In [99]:
# get arrays for training and test
X_train, y_train, X_test , y_test, means, std_devs = build_dataset(mc_awk)

# Building Neural Network

In [74]:
# Neural Network architecture

def build_baseline_nn(input_dim):
    """Creates a simple neural network, fully-connected and regularized"""

    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        layers.Dense(32, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        layers.Dense(1, activation="sigmoid")
    ])
    
    model.compile(
        optimizer=optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [ ]:
# Pipeline execution

# hyperparameters settings
epochs = 5
batch_size = 1024
validation_split = 0.15


if __name__ == "__main__":
        
    print(f"Train set shape: {X_train.shape}")
    print(f"Test set shape: {X_test.shape}")
    
    , input_dimension  = X_train.shape
    
    model = build_baseline_nn(input_dim=input_dimension)
    model.summary()
    
    print("\nBegin training...")
    history = model.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=validation_split,
        verbose=1
    )
    
    print("\nPerformance evaluation on Test set...")
    # batch_size specified in order to avoid memory peaks during prediction
    y_pred_prob = model.predict(X_test, batch_size=1024).ravel()
    y_pred = (y_pred_prob > 0.5).astype(int)
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    # Performance plots
    fig, (ax_loss, ax_acc, ax_roc) = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Loss plot
    ax_loss.plot(history.history["loss"], label="Train Loss", color="blue")
    ax_loss.plot(history.history["val_loss"], label="Val Loss", color="orange", linestyle="--")
    ax_loss.set_title("Loss function ")
    ax_loss.set_xlabel("Epochs")
    ax_loss.set_ylabel("Loss")
    ax_loss.legend()
    ax_loss.grid(True)
    
    # 2. Accuracy Plot
    ax_acc.plot(history.history["accuracy"], label="Train Acc", color="blue")
    ax_acc.plot(history.history["val_accuracy"], label="Val Acc", color="orange", linestyle="--")
    ax_acc.set_title("Accuracy")
    ax_acc.set_xlabel("Epochs")
    ax_acc.set_ylabel("Accuracy")
    ax_acc.legend()
    ax_acc.grid(True)
    
    # 3. ROC & AUC plot
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
    roc_auc = auc(fpr, tpr)
    
    ax_roc.plot(fpr, tpr, color="darkred", lw=2, label=f"ROC (AUC = {roc_auc:.4f})")
    ax_roc.plot(np.array((0, 1)), np.array((0, 1)), color="navy", linestyle="--", label="Random")
    ax_roc.set_xlim(np.array((0.0, 1.0)))
    ax_roc.set_ylim(np.array((0.0, 1.05)))
    ax_roc.set_xlabel("False Positive Rate")
    ax_roc.set_ylabel("True Positive Rate")
    ax_roc.set_title("Receiver Operating Characteristic (ROC)")
    ax_roc.legend()
    ax_roc.grid(True)
    
    plt.tight_layout()
    plt.show()
    plt.close('all')

Train set shape: (4315451, 14)
Test set shape: (1078863, 14)


ValueError: too many values to unpack (expected 1)